In [24]:
from pathlib import Path
import sys
import pandas as pd

In [ ]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.database import get_engine
from sqlalchemy import text
import os
import getpass

os.environ["PGUSER"] = "sgurung"
os.environ["PGDATABASE"] = "citibike"
os.environ["PGPASSWORD"] = getpass.getpass("PostgreSQL password: ")

In [ ]:
engine = get_engine()
with engine.connect() as conn:
        data = pd.read_sql(text(
            f"""
            Select *
            from neighborhood_hourly_model_data
            """
        ), con=conn)

data.head()

,hour,nta_code,nta_name,borough,departures,arrivals,net_flow,temperature,dew_point,precipitation,snowfall,weather_code,wind_speed
0,2024-12-30 23:00:00,QN0103,Astoria (Central),Queens,1.0,0.0,-1.0,7.2,1.40,0.0,0.0,0.0,11.465024
1,2024-12-31 00:00:00,BK0103,South Williamsburg,Brooklyn,1.0,0.0,-1.0,6.4,1.35,0.0,0.0,0.0,10.534229
2,2024-12-31 10:00:00,BK5591,Prospect Park,Brooklyn,1.0,0.0,-1.0,5.7,1.75,0.0,0.0,0.0,1.800000
3,2024-12-31 11:00:00,BK0101,Greenpoint,Brooklyn,1.0,0.0,-1.0,7.6,2.70,0.0,0.0,1.0,2.255393
4,2024-12-31 12:00:00,BK0102,Williamsburg,Brooklyn,1.0,0.0,-1.0,9.2,3.30,0.0,0.0,2.0,4.711050


In [27]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1510692 entries, 0 to 1510691
Data columns (total 13 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   hour           1510692 non-null  datetime64[ns]
 1   nta_code       1510692 non-null  object        
 2   nta_name       1510692 non-null  object        
 3   borough        1510692 non-null  object        
 4   departures     1510692 non-null  float64       
 5   arrivals       1510692 non-null  float64       
 6   net_flow       1510692 non-null  float64       
 7   temperature    1510692 non-null  float64       
 8   dew_point      1510692 non-null  float64       
 9   precipitation  1510692 non-null  float64       
 10  snowfall       1510692 non-null  float64       
 11  weather_code   1510692 non-null  float64       
 12  wind_speed     1510692 non-null  float64       
dtypes: datetime64[ns](1), float64(9), object(3)
memory usage: 149.8+ MB


In [28]:
from pandas.tseries.holiday import USFederalHolidayCalendar
holiday_calendar = USFederalHolidayCalendar()
holiday_calendar = holiday_calendar.holidays(start=data["hour"].min(),
                        end=data["hour"].max())
data["is_Holiday"] = data["hour"].isin(holiday_calendar)

display(holiday_calendar,
        data.loc[data["hour"].isin(holiday_calendar)])

DatetimeIndex(['2025-01-01', '2025-01-20', '2025-02-17', '2025-05-26',
               '2025-06-19', '2025-07-04', '2025-09-01', '2025-10-13',
               '2025-11-11', '2025-11-27', '2025-12-25', '2026-01-01',
               '2026-01-19', '2026-02-16', '2026-05-25', '2026-06-19',
               '2026-07-03'],
              dtype='datetime64[ns]', freq=None)

,hour,nta_code,nta_name,borough,departures,arrivals,net_flow,temperature,dew_point,precipitation,snowfall,weather_code,wind_speed,is_Holiday
97,2025-01-01,BK0101,Greenpoint,Brooklyn,20.0,30.0,10.0,9.05,8.30,0.9,0.0,53.0,12.425216,True
98,2025-01-01,BK0102,Williamsburg,Brooklyn,61.0,40.0,-21.0,9.05,8.30,0.9,0.0,53.0,12.425216,True
99,2025-01-01,BK0103,South Williamsburg,Brooklyn,0.0,2.0,2.0,9.05,8.30,0.9,0.0,53.0,12.425216,True
100,2025-01-01,BK0104,East Williamsburg,Brooklyn,25.0,19.0,-6.0,9.05,8.30,0.9,0.0,53.0,12.425216,True
101,2025-01-01,BK0201,Brooklyn Heights,Brooklyn,12.0,9.0,-3.0,9.05,8.30,0.9,0.0,53.0,12.425216,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428806,2026-07-03,QN0501,Maspeth,Queens,4.0,11.0,7.0,33.30,21.65,0.0,0.0,0.0,10.285232,True
1428807,2026-07-03,QN0502,Ridgewood,Queens,32.0,55.0,23.0,33.30,21.65,0.0,0.0,0.0,10.285232,True
1428808,2026-07-03,QN0503,Glendale,Queens,1.0,4.0,3.0,33.30,21.65,0.0,0.0,0.0,10.285232,True
1428809,2026-07-03,QN0601,Rego Park,Queens,1.0,1.0,0.0,33.30,21.65,0.0,0.0,0.0,10.285232,True


In [ ]:
engine = get_engine()
with engine.connect() as conn:
        data = pd.read_sql(text(
            f"""
            Select *
            from neighborhood_hourly_features
            """
        ), con=conn)

data.head()

,hour,nta_code,nta_name,borough,departures,arrivals,net_flow,temperature,dew_point,precipitation,...,departures_lag_504,departures_lag_672,departures_historical_average,departures_historical_std,arrivals_lag_168,arrivals_lag_336,arrivals_lag_504,arrivals_lag_672,arrivals_historical_average,arrivals_historical_std
0,2025-01-28 19:00:00,BK0102,Williamsburg,Brooklyn,270.0,256.0,-14.0,-2.95,-7.95,0.0,...,195.0,1.0,138.50,101.303175,126.0,218.0,168.0,0.0,128.00,93.252346
1,2025-01-28 20:00:00,BK0102,Williamsburg,Brooklyn,190.0,154.0,-36.0,-1.95,-7.75,0.0,...,115.0,2.0,98.25,72.811972,86.0,163.0,91.0,0.0,85.00,66.698326
2,2025-01-28 21:00:00,BK0102,Williamsburg,Brooklyn,112.0,118.0,6.0,-1.60,-8.25,0.0,...,76.0,1.0,60.75,45.257596,43.0,83.0,56.0,0.0,45.50,34.607321
3,2025-01-28 22:00:00,BK0102,Williamsburg,Brooklyn,89.0,82.0,-7.0,-1.00,-6.95,0.0,...,53.0,1.0,45.75,38.724454,29.0,70.0,40.0,0.0,34.75,28.929512
4,2025-01-28 22:00:00,MN0401,Chelsea-Hudson Yards,Manhattan,140.0,140.0,0.0,-1.00,-6.95,0.0,...,95.0,1.0,66.00,45.328431,65.0,95.0,78.0,0.0,59.50,41.525093


In [30]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1426536 entries, 0 to 1426535
Data columns (total 33 columns):
 #   Column                         Non-Null Count    Dtype         
---  ------                         --------------    -----         
 0   hour                           1426536 non-null  datetime64[ns]
 1   nta_code                       1426536 non-null  object        
 2   nta_name                       1426536 non-null  object        
 3   borough                        1426536 non-null  object        
 4   departures                     1426536 non-null  float64       
 5   arrivals                       1426536 non-null  float64       
 6   net_flow                       1426536 non-null  float64       
 7   temperature                    1426536 non-null  float64       
 8   dew_point                      1426536 non-null  float64       
 9   precipitation                  1426536 non-null  float64       
 10  snowfall                       1426536 non-null  float

In [31]:
data.isna().sum()

hour                             0
nta_code                         0
nta_name                         0
borough                          0
departures                       0
arrivals                         0
net_flow                         0
temperature                      0
dew_point                        0
precipitation                    0
snowfall                         0
weather_code                     0
wind_speed                       0
year                             0
month                            0
hour_of_day                      0
day_of_week                      0
is_Holiday                       0
temperature_lag_1hr              0
precipitation_lag_1hr            0
snowfall_lag_1hr                 0
departures_lag_168               0
departures_lag_336               0
departures_lag_504               0
departures_lag_672               0
departures_historical_average    0
departures_historical_std        0
arrivals_lag_168                 0
arrivals_lag_336    